In [5]:
# ============================================================
# Monthly BFI with 9 baseflow separation methods 
# ============================================================

import warnings
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from baseflow.methods import (
    LH,
    UKIH,
    Chapman,
    CM,
    Boughton,
    Furey,
    Eckhardt,
    EWMA,
    Willems,
)
from baseflow.comparision import strict_baseflow
from baseflow.utils import clean_streamflow, exist_ice
from baseflow.param_estimate import recession_coefficient, param_calibrate


METHODS_9: List[str] = [
    "UKIH",
    "LH",
    "Chapman",
    "CM",
    "Boughton",
    "Furey",
    "Eckhardt",
    "EWMA",
    "Willems",
]


def _clip_baseflow(b: np.ndarray, Q: np.ndarray) -> np.ndarray:
    b = np.asarray(b, dtype=float)
    Q = np.asarray(Q, dtype=float)

    b = np.where(np.isfinite(b), b, np.nan)
    b = np.where(b < 0, 0.0, b)

    ok = np.isfinite(Q) & (Q >= 0)
    b[ok] = np.minimum(b[ok], Q[ok])
    return b


def single_station_baseflow_9(series: pd.Series, ice: Optional[object] = None) -> pd.DataFrame:
    date, Q = clean_streamflow(series)

    ice_bool = exist_ice(date, ice) if ice is not None else None
    strict = strict_baseflow(Q, ice_bool)

    a = recession_coefficient(Q, strict)  # needed for several methods
    b_LH = LH(Q)

    out = pd.DataFrame(index=date, columns=METHODS_9, dtype=float)

    for m in METHODS_9:
        if m == "UKIH":
            b = UKIH(Q, b_LH)

        elif m == "LH":
            b = b_LH

        elif m == "Chapman":
            b = Chapman(Q, b_LH, a)

        elif m == "CM":
            b = CM(Q, b_LH, a)

        elif m == "Boughton":
            C = param_calibrate(np.arange(0.0001, 0.1, 0.0001), Boughton, Q, b_LH, a)
            b = Boughton(Q, b_LH, a, C)

        elif m == "Furey":
            A = param_calibrate(np.arange(0.01, 10, 0.01), Furey, Q, b_LH, a)
            b = Furey(Q, b_LH, a, A)

        elif m == "Eckhardt":
            BFImax = param_calibrate(np.arange(0.001, 1, 0.001), Eckhardt, Q, b_LH, a)
            b = Eckhardt(Q, b_LH, a, BFImax)

        elif m == "EWMA":
            e = param_calibrate(np.arange(0.0001, 0.1, 0.0001), EWMA, Q, b_LH, 0)
            b = EWMA(Q, b_LH, 0, e)

        elif m == "Willems":
            w = param_calibrate(np.arange(0.001, 1, 0.001), Willems, Q, b_LH, a)
            b = Willems(Q, b_LH, a, w)

        out[m] = _clip_baseflow(b, Q)

    return out


def monthly_bfi_from_daily(Q_daily: pd.Series, b_daily: pd.DataFrame, min_days: int = 15) -> pd.DataFrame:
    Qv = pd.to_numeric(Q_daily, errors="coerce").astype(float)
    Qv = Qv.where(Qv >= 0)

    b_daily = b_daily.reindex(Qv.index)

    Qm = Qv.resample("MS").sum(min_count=1)
    bm = b_daily.resample("MS").sum(min_count=1)

    cnt = Qv.resample("MS").count()
    bfi = bm.div(Qm, axis=0)

    bfi = bfi.where(cnt >= min_days).clip(0, 1)
    return bfi


def plot_monthly_bfi(station: str, bfi_monthly: pd.DataFrame, out_png: Path, show: bool = False) -> None:
    plt.figure()
    for m in bfi_monthly.columns:
        plt.plot(bfi_monthly.index, bfi_monthly[m], label=m)
    plt.ylim(0, 1)
    plt.title(f"Monthly BFI comparison — {station}")
    plt.xlabel("Month")
    plt.ylabel("BFI")
    plt.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    if show:
        plt.show()
    plt.close()


def plot_seasonal_cycle(station: str, bfi_monthly: pd.DataFrame, out_png: Path, show: bool = False) -> None:
    clim = bfi_monthly.copy()
    clim["month"] = clim.index.month
    clim = clim.groupby("month").mean(numeric_only=True)

    plt.figure()
    for m in [c for c in clim.columns if c in METHODS_9]:
        plt.plot(clim.index, clim[m], marker="o", label=m)
    plt.ylim(0, 1)
    plt.xticks(range(1, 13))
    plt.title(f"BFI seasonal cycle (mean across years) — {station}")
    plt.xlabel("Calendar month")
    plt.ylabel("BFI")
    plt.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    if show:
        plt.show()
    plt.close()


def run_monthly_bfi_pipeline(
    csv_path: str,
    out_dir: str,
    time_col: str = "time",
    min_days_per_month: int = 15,
    plot_stations: Optional[List[str]] = None,
    show_plots: bool = False,
    ice: Optional[object] = None,
) -> None:
    outdir = Path(out_dir)
    outdir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(csv_path)

    if time_col not in df.columns:
        raise ValueError(f"Expected '{time_col}' column. Found: {list(df.columns)}")

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce", infer_datetime_format=True)
    df = df.dropna(subset=[time_col]).sort_values(time_col)

    dfQ = df.set_index(time_col)
    for c in dfQ.columns:
        dfQ[c] = pd.to_numeric(dfQ[c], errors="coerce")

    stations = [str(s).strip() for s in dfQ.columns]
    dfQ.columns = stations  # normalize column names

    print(f"Found {len(stations)} stations. First 10: {stations[:10]}")

    # Choose which stations to plot
    if plot_stations is None:
        plot_stations = stations[:2]  # default: first 2
    else:
        plot_stations = [str(s).strip() for s in plot_stations]

    # Fallback if user-requested station names do not match
    missing = [s for s in plot_stations if s not in stations]
    if missing:
        print(f"WARNING: These plot_stations were not found in columns: {missing}")
        print("Falling back to plotting the first station in the file.")
        plot_stations = [stations[0]]

    long_rows = []

    for sta in tqdm(stations, desc="Stations", total=len(stations)):
        series = dfQ[sta]

        try:
            b_daily = single_station_baseflow_9(series, ice=ice)
            Q_aligned = series.reindex(b_daily.index)
            bfi_m = monthly_bfi_from_daily(Q_aligned, b_daily, min_days=min_days_per_month)

            out_station_csv = outdir / f"BFI_monthly_{sta}.csv"
            bfi_m.to_csv(out_station_csv)

            tmp = bfi_m.reset_index().rename(columns={"index": "month"})
            tmp.insert(1, "station", sta)
            long = tmp.melt(id_vars=["month", "station"], var_name="method", value_name="BFI")
            long_rows.append(long)

            if sta in plot_stations:
                p1 = outdir / f"BFI_monthly_{sta}.png"
                p2 = outdir / f"BFI_seasonalcycle_{sta}.png"
                plot_monthly_bfi(sta, bfi_m, p1, show=show_plots)
                plot_seasonal_cycle(sta, bfi_m, p2, show=show_plots)
                print(f"Saved plots:\n  {p1}\n  {p2}")

        except Exception as e:
            warnings.warn(f"Failed station {sta}: {e}")
            continue

    if long_rows:
        all_long = pd.concat(long_rows, ignore_index=True)
        out_long = outdir / "BFI_monthly_9methods_long.csv"
        all_long.to_csv(out_long, index=False)
        print(f"Saved combined long table: {out_long}")

    print(f"Done. Outputs in: {outdir}")


if __name__ == "__main__":
    CSV_PATH = r"C:\Users\Lenovo\Downloads\example.csv"
    OUT_DIR  = r"C:\Users\Lenovo\Downloads"

    run_monthly_bfi_pipeline(
        csv_path=CSV_PATH,
        out_dir=OUT_DIR,
        time_col="time",
        min_days_per_month=15,
        plot_stations=["GRDC_1160815"],  # must match exactly 
        show_plots=True,                 
        ice=None,
    )


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32580\4031430441.py:175: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce", infer_datetime_format=True)


Found 6 stations. First 10: ['GRDC_1160815', 'US_09447000', 'GRDC_1160822', 'US_09447600', 'GRDC_1190815', 'US_094887000']


Stations:   0%|          | 0/6 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32580\4031430441.py:226: UserWarning: Failed station GRDC_1160815: "The following id_vars or value_vars are not present in the DataFrame: ['month']"
  warnings.warn(f"Failed station {sta}: {e}")
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32580\4031430441.py:226: UserWarning: Failed station US_09447000: "The following id_vars or value_vars are not present in the DataFrame: ['month']"
  warnings.warn(f"Failed station {sta}: {e}")
Stations:  33%|███▎      | 2/6 [00:00<00:00, 18.32it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32580\4031430441.py:226: UserWarning: Failed station GRDC_1160822: "The following id_vars or value_vars are not present in the DataFrame: ['month']"
  warnings.warn(f"Failed station {sta}: {e}")
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32580\4031430441.py:226: UserWarning: Failed station US_09447600: "The following id_vars or value_vars are not present in the DataFram

Done. Outputs in: C:\Users\Lenovo\Downloads
